In [1]:
# ============================================
# Experiment harness for ON/OFF window models
# ============================================
import numpy as np, pandas as pd
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, List
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, f1_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA


In [ ]:
win_task1 = pd.read_csv("")
win_task2 = pd.read_csv("outputs/windows_task2.csv")
win_task3 = pd.read_csv("outputs/windows_task3.csv")

In [ ]:
# ------- defaults you can tweak ----------
FEATS_BASE = [
    "fix_efficiency","fix_aoi_ratio",
    "gaze_dispersion_total",
    "pupil_mean","pupil_var",
    "saccade_mean_mag","saccade_dir_var",
    "blink_rate","valid_gaze_ratio",
]

LABEL_COL = "label"
GROUP_COL = "User ID (file)"        # grouping by participant
TASK_ID_COLS = ["Video Name (folder)","User ID (file)","Task Id"]  # ID triple

# ----------------- config -----------------
@dataclass
class ExpConfig:
    name: str
    feats: List[str] = None
    qc_min_valid: float = 0.0            # drop windows with valid_gaze_ratio < this
    drop_na_rows: bool = True            # drop rows with any NaN in selected feats after impute? (imputer handles most)
    model: str = "LR"                    # "LR" or "RF"
    class_weight: Optional[str] = None   # None or "balanced"  (LR & RF support)
    lr_C: float = 1.0                    # LR inverse regularization strength
    rf_n_estimators: int = 400
    rf_max_depth: Optional[int] = None
    pca_n: Optional[int] = None          # e.g., 5 (applied inside pipeline, before model)
    n_splits: int = 5                    # GroupKFold splits by participant
    random_state: int = 0

    def ensure(self):
        if self.feats is None:
            self.feats = FEATS_BASE

# ------------- helpers --------------------
def _prep_dataframe(df: pd.DataFrame, cfg: ExpConfig) -> pd.DataFrame:
    d = df.copy()
    d[LABEL_COL] = d[LABEL_COL].astype(str).str.upper().str.strip()
    d = d[d[LABEL_COL].isin(["ON","OFF"])].copy()
    # QC filter
    if cfg.qc_min_valid > 0 and "valid_gaze_ratio" in d.columns:
        d = d[d["valid_gaze_ratio"] >= cfg.qc_min_valid].copy()
    # ensure IDs exist for later pairing/reporting (not strictly needed here)
    for c in TASK_ID_COLS:
        if c not in d.columns: d[c] = "UNK"
    return d

def _build_pipeline(cfg: ExpConfig):
    steps = [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
    if cfg.pca_n and cfg.pca_n > 0:
        steps.append(("pca", PCA(n_components=cfg.pca_n, random_state=cfg.random_state)))
    if cfg.model.upper() == "LR":
        clf = LogisticRegression(
            penalty="l2", C=cfg.lr_C, solver="liblinear",
            max_iter=1000, class_weight=cfg.class_weight, random_state=cfg.random_state
        )
    elif cfg.model.upper() == "RF":
        clf = RandomForestClassifier(
            n_estimators=cfg.rf_n_estimators,
            max_depth=cfg.rf_max_depth,
            random_state=cfg.random_state,
            class_weight=cfg.class_weight,
            n_jobs=-1
        )
    else:
        raise ValueError("model must be 'LR' or 'RF'")
    steps.append(("clf", clf))
    return Pipeline(steps)

def _fold_metrics(y_true, y_prob, y_pred):
    # positive class = "ON"
    auroc  = roc_auc_score(y_true, y_prob)
    auprc  = average_precision_score((y_true==1).astype(int), y_prob)
    balacc = balanced_accuracy_score(y_true, y_pred)
    macrof1= f1_score(y_true, y_pred, average="macro")
    brier  = brier_score_loss((y_true==1).astype(int), y_prob)
    return dict(AUROC=auroc, AUPRC_ON=auprc, BalancedAcc=balacc, MacroF1=macrof1, Brier=brier)

# ---------------- core runner ----------------
def run_experiment(df: pd.DataFrame, cfg: ExpConfig) -> pd.DataFrame:
    """
    Returns a per-fold metrics DataFrame with a final 'mean' row.
    Uses GroupKFold by participant for comparability across runs (same seed -> same groups).
    """
    cfg.ensure()
    d = _prep_dataframe(df, cfg)
    if d.empty:
        raise ValueError("No ON/OFF rows after preprocessing.")

    X = d[cfg.feats].copy()
    y = (d[LABEL_COL] == "ON").astype(int).values
    groups = d[GROUP_COL].astype(str).values

    pipe = _build_pipeline(cfg)

    # Fix splits deterministically by sorting groups
    unique_groups = np.array(sorted(np.unique(groups)))
    # Create indices per group for GroupKFold
    gkf = GroupKFold(n_splits=min(cfg.n_splits, len(unique_groups)))
    fold_rows = []
    fold_id = 1
    for tr_idx, va_idx in gkf.split(X, y, groups):
        Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        pipe.fit(Xtr, ytr)
        ypr = pipe.predict_proba(Xva)[:,1]
        yhat = (ypr >= 0.5).astype(int)
        m = _fold_metrics(yva, ypr, yhat)
        m.update(dict(fold=fold_id, n_val=len(va_idx)))
        fold_rows.append(m)
        fold_id += 1

    res = pd.DataFrame(fold_rows)
    mean = res.mean(numeric_only=True).to_dict()
    std  = res.std(numeric_only=True).to_dict()
    mean.update({k+"_std": std.get(k, np.nan) for k in mean.keys()})
    mean["fold"] = "mean"
    res = pd.concat([res, pd.DataFrame([mean])], ignore_index=True)
    res.insert(0, "exp_name", cfg.name)
    res.insert(1, "model", cfg.model.upper())
    res.insert(2, "feats", [",".join(cfg.feats)] + [""]*(len(res)-1))
    res.insert(3, "qc_min_valid", cfg.qc_min_valid)
    res.insert(4, "pca_n", cfg.pca_n if cfg.pca_n else 0)
    res.insert(5, "class_weight", cfg.class_weight if cfg.class_weight else "None")
    return res

# --------------- multi-run convenience ---------------
def run_ablation(df: pd.DataFrame, configs: List[ExpConfig]) -> pd.DataFrame:
    tables = []
    for cfg in configs:
        out = run_experiment(df, cfg)
        tables.append(out)
        # brief console summary
        row = out[out["fold"]=="mean"].iloc[0]
        print(f"[{cfg.name}] {cfg.model}: AUROC={row['AUROC']:.3f}±{row['AUROC_std']:.3f}  "
              f"BalAcc={row['BalancedAcc']:.3f}  MacroF1={row['MacroF1']:.3f}  Brier={row['Brier']:.3f}")
    return pd.concat(tables, ignore_index=True)

# ============================
# Example usage (per task)
# ============================
# Suppose you already have: win_task1, win_task2, win_task3
# Start with a frozen baseline, then tweak ONE thing per config.

baseline = ExpConfig(
    name="baseline_LR",
    feats=FEATS_BASE,
    qc_min_valid=0.5,      # baseline QC
    model="LR",
    class_weight=None,
    lr_C=1.0,
    pca_n=None,
    n_splits=5,
    random_state=0
)

tweak_qc = ExpConfig(**{**asdict(baseline), "name":"qc0.7_LR", "qc_min_valid":0.7})
tweak_pca= ExpConfig(**{**asdict(baseline), "name":"pca5_LR", "pca_n":5})
rf_base  = ExpConfig(**{**asdict(baseline), "name":"baseline_RF", "model":"RF", "class_weight":"balanced"})

# Run on one task (repeat for others)
# ablation1 = run_ablation(win_task1, [baseline, tweak_qc, tweak_pca, rf_base])
# display(ablation1[ablation1["fold"]=="mean"].sort_values("AUROC", ascending=False))